# 5 — Critic Loop Agent (Summarizer ↔ Reviewer)

Two LLM agents check each other's work: a summarizer drafts, a reviewer critiques, the summarizer revises — cycling until quality is met or iterations cap out.

**What you'll learn**
- `Annotated[list[AnyMessage], operator.add]` — state that accumulates messages rather than replacing them
- The critic loop pattern: `summarizer → reviewer → summarizer` until stopping condition
- `should_continue` with iteration counting — how to break a cycle after N rounds
- Why the reviewer's critique must be in `state["messages"]` for the summarizer to read it
- `MemorySaver` + `thread_id` for multi-turn refinement

**Companion examples:** 4-basic-react-agent (simpler single-agent loop), 18-self-reflecting-agent (agent critiques its own output)

In [ ]:
# ── 1. Install dependencies (Colab only) ──────────────────────────────────────
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q langchain langchain-openai python-dotenv langgraph

In [ ]:
# ── 2. API key ─────────────────────────────────────────────────────────────────
import os

from dotenv import load_dotenv

if not IN_COLAB:
    load_dotenv()

if IN_COLAB and not os.environ.get("OPENAI_API_KEY"):
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running"

In [ ]:
# ── 3. Why Annotated[list, operator.add] matters here ─────────────────────────
# A normal TypedDict field gets REPLACED each time a node returns it.
# With operator.add, each return is APPENDED to the existing list.
#
# For the critic loop this is essential:
# If messages were replaced, the reviewer's critique would disappear when
# the summarizer writes its revision — the loop would lose context.
import operator

existing = ["AIMessage: first draft"]
from_reviewer = ["AIMessage: needs more specific numbers"]
merged = operator.add(existing, new=from_reviewer)
print("After operator.add:", merged)
print("-> Summarizer sees BOTH messages on its next turn")

In [ ]:
# ── 4. Build the critic loop graph ────────────────────────────────────────────
# The original script uses PyPDFLoader to load a product spec PDF.
# We use inline text to stay self-contained — the graph pattern is identical.
import operator
import uuid
from typing import Annotated, TypedDict

from langchain_core.messages import AnyMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, StateGraph

SOURCE_TEXT = (
    "EcoSprint X3 Specification\n"
    "The EcoSprint X3 is a lightweight electric bicycle for urban commuters. "
    "Motor: 250W brushless mid-drive. Battery: 36V 10Ah lithium-ion, range 60-80km. "
    "Frame: 6061 aluminum, weight 18kg. Brakes: hydraulic disc front and rear. "
    "Display: backlit LCD. Charging: 3.5 hours. Max speed: 25 km/h. "
    "Foldable. IP54 water resistance. Price: EUR 1,299."
)

SUMMARIZER_PROMPT = (
    "You are a document summarizer. Create a summary in under 50 words. "
    "If the user has provided critique, respond with a revised version of your previous summary."
)

REVIEWER_PROMPT = (
    "You are a quality reviewer. Compare the source document and the generated summary. "
    "Check accuracy and completeness. Give improvement recommendations in under 50 words."
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)


class SummaryAgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]


def generate_summary(state: SummaryAgentState) -> dict:
    messages = [SystemMessage(SUMMARIZER_PROMPT)] + state["messages"]
    result = llm.invoke(messages)
    print(f"  [summarizer] {result.content[:100]}")
    return {"messages": [result]}


def review_summary(state: SummaryAgentState) -> dict:
    messages = [SystemMessage(REVIEWER_PROMPT)] + state["messages"]
    result = llm.invoke(messages)
    print(f"  [reviewer]   {result.content[:100]}")
    return {"messages": [result]}


def should_continue(state: SummaryAgentState) -> bool:
    # 1 input + 2 messages per round. Stop after 2 rounds (5 total messages).
    return len(state["messages"]) <= 4


graph = StateGraph(SummaryAgentState)
graph.add_node("summarizer", generate_summary)
graph.add_node("reviewer", review_summary)
graph.add_conditional_edges("summarizer", should_continue, {True: "reviewer", False: END})
graph.add_edge("reviewer", "summarizer")
graph.set_entry_point("summarizer")
app = graph.compile(checkpointer=MemorySaver())

print("Graph: summarizer <-> reviewer (loops until should_continue=False)")

In [ ]:
# ── 5. Run the critic loop ─────────────────────────────────────────────────────
config = {"configurable": {"thread_id": str(uuid.uuid4())}}

print("Starting critic loop...\n")
result = app.invoke({"messages": [HumanMessage(SOURCE_TEXT)]}, config)

print(f"\n=== Final summary ({len(result['messages'])} messages total) ===")
# The last AIMessage is from the summarizer (graph ends on summarizer node)
from langchain_core.messages import AIMessage

for msg in reversed(result["messages"]):
    if isinstance(msg, AIMessage):
        print(msg.content)
        break

In [ ]:
# ── 6. Multi-turn refinement — user directs revisions ─────────────────────────
# Same thread_id keeps the conversation history.
# The user can steer the summary without re-running from scratch.
config2 = {"configurable": {"thread_id": str(uuid.uuid4())}}

# Initial run
r = app.invoke({"messages": [HumanMessage(SOURCE_TEXT)]}, config2)
print("Initial summary:\n", r["messages"][-1].content, "\n")

# User feedback turn 1
r = app.invoke({"messages": [HumanMessage("Focus more on the battery specs and range.")]}, config2)
print("After battery focus:\n", r["messages"][-1].content)

## Exercises

**Exercise 1 — Change the stopping condition:** Change `should_continue` to stop after 1 round (`len <= 2`). Does summary quality change? Is one review enough?

**Exercise 2 — Different document:** Replace `SOURCE_TEXT` with a paragraph from any article. Does the reviewer catch genuine inaccuracies?

**Exercise 3 — Stricter reviewer:** Change the reviewer prompt to: `"Reject summaries unless they include at least 3 specific numbers or measurements."` How does the summary change?

**Exercise 4 — Log each round:** Add a round counter to `SummaryAgentState` and increment it in each node. Print which round each message belongs to.

## What's next

- **4-basic-react-agent** — single-agent tool calling without a critic loop
- **6-multi-agent-graph** — routing between specialist agents based on query type
- **18-self-reflecting-agent** — reflexion: an agent critiques its own output